Training a Recurrent Neural Network Character by Character with Shakespeare text.

In [1]:
from pathlib import Path
import urllib.request

In [2]:
def download_shakespeare_text():
    path = Path("datasets/shakespeare/shakespeare.txt")
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        url ="https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        urllib.request.urlretrieve(url, path)
    return path.read_text()

shakespeare_text = download_shakespeare_text()


In [3]:
print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [4]:
vocab = sorted(set(shakespeare_text.lower()))
"".join(vocab)

"\n !$&',-.3:;?abcdefghijklmnopqrstuvwxyz"

In [5]:
char_to_id = {char: index for index, char in enumerate(vocab)}
id_to_char = {index: char for index, char in enumerate(vocab)}

In [6]:
import torch

def encode_text(text):
    return torch.tensor([char_to_id[char] for char in text.lower()])

def decode_text(char_ids):
    return "".join([id_to_char[char_id.item()] for char_id in char_ids])

In [7]:
encoded = encode_text("Hello, world!")
encoded

tensor([20, 17, 24, 24, 27,  6,  1, 35, 27, 30, 24, 16,  2])

In [8]:
decode_text(encoded)

'hello, world!'

In [9]:
from torch.utils.data import Dataset, DataLoader
class CharDataset(Dataset):

    def __init__(self, text, window_length):
        self.encoded_text = encode_text(text)
        self.window_length = window_length

    def __len__(self):
        return len(self.encoded_text) - self.window_length
    
    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        
        end = idx + self.window_length
        window = self.encoded_text[idx:end]
        target = self.encoded_text[idx+1 : end+1]
        return window, target

In [ ]:
window_length = 25 # reduce to faster training
batch_size = 256 # reduce if your GPU cannot handle such a large batch size

train_set = CharDataset(shakespeare_text[:1_000_000], window_length)
valid_set = CharDataset(shakespeare_text[1_000000:1_060_000], window_length)
test_set = CharDataset(shakespeare_text[1_060_000:], window_length)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)

Model and Embedding

In [11]:
import torch.nn as nn
torch.manual_seed(42)

In [12]:
class ShakespeareModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=10, hidden_dim=128,
                 dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, X):
        embeddings = self.embed(X)
        outputs, _states = self.gru(embeddings)
        return self.output(outputs).permute(0, 2, 1)
    

In [13]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

mps


In [14]:
model = ShakespeareModel(len(vocab)).to(device)

In [ ]:

import torch.nn as nn
import torch.optim as optim
import torchmetrics

# Loss function:ross‑entropy
criterion = nn.CrossEntropyLoss()

# Optimizer:Adam
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Metric for validation accuracy
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=len(vocab)).to(device)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        y_pred = model(X_batch)                     
        loss = criterion(y_pred, y_batch)           

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, accuracy_metric):
    model.eval()
    total_loss = 0.0
    accuracy_metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()

            preds = y_pred.argmax(dim=1)           
            accuracy_metric.update(preds, y_batch)
    avg_loss = total_loss / len(loader)
    acc = accuracy_metric.compute()
    return avg_loss, acc

n_epochs = 15   # train more if you can
for epoch in range(n_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, valid_loader, criterion, accuracy)

    print(f"Epoch {epoch+1:2d}/{n_epochs} | "
          f"Train loss: {train_loss:.4f} | "
          f"Valid loss: {val_loss:.4f} | "
          f"Valid acc: {val_acc:.4f}")

Epoch  1/3 | Train loss: 1.5965 | Valid loss: 1.5261 | Valid acc: 0.5394
Epoch  2/3 | Train loss: 1.3942 | Valid loss: 1.5004 | Valid acc: 0.5463


KeyboardInterrupt: 

In [ ]:
model.eval()
test_loss, test_acc = evaluate(model, test_loader, criterion, accuracy)

print(f"Final test loss: {test_loss:.4f} | test accuracy: {test_acc:.4f}")

In [ ]:
 text = "To be or not to b"
encoded_text = encode_text(text).unsqueeze(dim=0).to(device)

with torch.no_grad():
    Y_logits = model(encoded_text)
    predicted_char_id = Y_logits[0, :,-1].argmax().item()
    predicted_char = id_to_char[predicted_char_id]

In [18]:
predicted_char

'e'

In [19]:
import torch.nn.functional as F

def next_char(model, text, temperature = 1):
    encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
    with torch.no_grad():
        Y_logits = model(encoded_text)
        Y_probas = F.softmax(Y_logits[0, :, -1] / temperature, dim=-1)
        predicted_char_id = torch.multinomial(Y_probas, num_samples=1).item()

    return id_to_char[predicted_char_id]

In [20]:
def extend_text(model, text, n_chars=80, temperature=1):
    for _ in range(n_chars):
        text += next_char(model, text, temperature)
    return text

In [21]:
print(extend_text(model, "To be or not to b", temperature=0.4))

To be or not to be to death,
who come and the time to the time to rest thou stand and made
and th
